# 03 Phase Prediction Model

This notebook trains baseline machine-learning models to predict SBF-induced phase composition in spray-pyrolyzed apatite-wollastonite glass-ceramics.

The predictive task is:

```text
Sintering temperature + SBF soaking time + initial phase composition
↓
SBF phase composition
```

This notebook is intended as an interpretable proof-of-concept materials informatics workflow.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT_DIR = Path('..')
RAW_DATA_DIR = ROOT_DIR / 'data' / 'raw'
PROCESSED_DATA_DIR = ROOT_DIR / 'data' / 'processed'
FIGURES_DIR = ROOT_DIR / 'figures'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load and Prepare Data

The initial phase-composition dataset and SBF phase-evolution dataset are merged using sintering temperature.

In [ ]:
initial_df = pd.read_csv(RAW_DATA_DIR / 'initial_phase_composition.csv')
sbf_df = pd.read_csv(RAW_DATA_DIR / 'sbf_phase_evolution.csv')

initial_df = initial_df.rename(columns={
    'amorphous_pct': 'initial_amorphous_pct',
    'wollastonite_pct': 'initial_wollastonite_pct',
    'hydroxyapatite_pct': 'initial_hydroxyapatite_pct',
    'whitlockite_pct': 'initial_whitlockite_pct'
})

sbf_df = sbf_df.rename(columns={
    'amorphous_pct': 'sbf_amorphous_pct',
    'wollastonite_pct': 'sbf_wollastonite_pct',
    'hydroxyapatite_pct': 'sbf_hydroxyapatite_pct',
    'whitlockite_pct': 'sbf_whitlockite_pct'
})

ml_df = sbf_df.merge(initial_df, on='temperature_C', how='left')

ml_df['initial_crystallinity_pct'] = 100 - ml_df['initial_amorphous_pct']
ml_df['sbf_crystallinity_pct'] = 100 - ml_df['sbf_amorphous_pct']
ml_df['hydroxyapatite_gain_pct'] = ml_df['sbf_hydroxyapatite_pct'] - ml_df['initial_hydroxyapatite_pct']
ml_df['wollastonite_change_pct'] = ml_df['sbf_wollastonite_pct'] - ml_df['initial_wollastonite_pct']

ml_df.to_csv(PROCESSED_DATA_DIR / 'awgc_ml_dataset.csv', index=False)

ml_df.head()

## Define Features and Targets

The features represent processing conditions and initial material state. The targets represent phase composition after SBF immersion.

In [ ]:
feature_columns = [
    'temperature_C',
    'soaking_day',
    'initial_amorphous_pct',
    'initial_wollastonite_pct',
    'initial_hydroxyapatite_pct',
    'initial_whitlockite_pct',
    'initial_crystallinity_pct'
]

target_columns = [
    'sbf_amorphous_pct',
    'sbf_wollastonite_pct',
    'sbf_hydroxyapatite_pct',
    'sbf_whitlockite_pct'
]

X = ml_df[feature_columns]
y = ml_df[target_columns]

X.shape, y.shape

## Train-Test Split

Because this is a small experimental dataset, the results should be interpreted as a proof of concept, not as a universal predictive model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

## Train Baseline Models

Two baseline models are trained:

- Linear Regression
- Random Forest Regression

Linear regression provides a simple baseline, while random forest regression can capture nonlinear relationships.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df

## Predicted vs Experimental Phase Fractions

The following plot compares experimental and predicted phase fractions using the random forest model.

In [ ]:
best_model_name = 'Random Forest'
y_pred = predictions[best_model_name]

plt.figure(figsize=(6, 6))
plt.scatter(y_test.values.flatten(), y_pred.flatten())

min_val = min(y_test.values.flatten().min(), y_pred.flatten().min())
max_val = max(y_test.values.flatten().max(), y_pred.flatten().max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

plt.xlabel('Experimental phase fraction (%)')
plt.ylabel('Predicted phase fraction (%)')
plt.title('Predicted vs Experimental Phase Fractions')
plt.tight_layout()

plt.savefig(FIGURES_DIR / 'predicted_vs_experimental_phase_fractions.png', dpi=300)
plt.show()

## Interpretation

This model demonstrates a first machine-learning workflow for predicting SBF-induced phase composition from processing variables and initial phase state.

Because the dataset is small, the model is best interpreted as a transparent proof of concept. Future work should include additional experimental samples, raw XRD spectra, SEM images, and ion-release data.